## Project: Deutsch–Jozsa Algorithm

**Instructor:** Hoa Nguyen, Data61 at CSIRO

### Project overview

The **Deutsch–Jozsa algorithm** (1992) was the first quantum algorithm to demonstrate an **exponential speedup** over classical computation. In this project you will **implement** and **analyse** the **generalised Deutsch–Jozsa algorithm** for **n-qubit black-box functions**.

**Oracle:** Input: n-bit string → Output: 0 or 1 (black box).

### Deliverables

1. **Working implementation** of the **n-qubit Deutsch–Jozsa algorithm**
2. **Multiple oracle functions** demonstrating **constant** and **balanced** cases
3. **Analysis** of why the algorithm works via the **phase kickback** mechanism
4. **Analysis** of **circuit depth** and **gate complexity** as **n** scales
5. **Comparison** of quantum query complexity vs classical approaches
6. **Testing and validation** on ideal and noisy quantum simulators

Each **Deliverable** below includes a task and a **sample solution**. Work through them in order. All circuits run on **simulators** (ideal and noisy).

#### **Note**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   

Code snippets will be provided for you, except in certain cells where your task will be to complete the code to ensure our quantum program runs smoothly. These cells will be clearly marked with a comment like: <span style="font-family: monospace; font-weight: bold; color: #111; background-color: #fff8dc; padding: 2px 6px; border-radius: 4px;"> ### WRITE YOUR CODE BELOW THIS CELL ### </span>. Please also note that we’ll be running our quantum programs exclusively on simulators. No worries—these work perfectly well for our learning purposes.

Each **Deliverable** has a task and a **sample solution**. Complete tasks in order; we use **simulators** only (ideal and noisy). It is considered a foundational stepping stone for exploring more advanced quantum algorithms. We will begin by revisiting the problem that this algorithm is designed to solve, then construct our own examples of a balanced function and a constant function, and finally apply the algorithm to uncover the answer to the problem.

</div>

## **Pre-checking and Imports**


<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   
You should check your Qiskit version before starting the lab. It is recommended to use Qiskit version 2.0 or higher for the best experience.
</div>

In [ ]:
# If you run this notebook on Google Colab, please uncomment the following line and run it first to install the required packages
# !pip install -r https://raw.githubusercontent.com/hoaiocom/rmit-qss-2026/main/requirements.txt

# If you run on local machine, please follow the instructions in the README.md file to setup the virtual environment and install the required packages

In [ ]:
import qiskit
print(f"Qiskit version: {qiskit.__version__}")

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.visualization import plot_histogram
import random as rd

## **Problem**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   

We are given a mysterious function $f(x)$ that takes as input either a single bit or a string of binary bits and returns only one of two possible values: $0$ or $1$.
This mysterious function can be either constant or balanced:

- **Constant function**: always outputs $0$ or always outputs $1$, regardless of the input.

- **Balanced function**: where it always outputs exactly $2^{n-1}$ times $1$ and $2^{n-1}$ times $0$ for inputs of length $n$.
  
**Our goal** is to determine whether the mysterious function above is **a constant function** or **a balanced function**.

</div>

### **Classical Approach**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   
  
**In classical computation**, in the worst case, we need to query the hidden function $2^{n-1}+1$ times, where $n$ is the length of the binary input string. To make this clearer, we present the following example:

Suppose our input consists of $3$ bits, which gives us a total of $8$ possible input cases. In the classical approach, we must call the function $2^{3-1}+1 = 5$ times to determine whether it is constant or balanced. Imagine that for the first $4$ function calls, we supply different input bits and obtain the output bits shown in the table below. 

| Input (3 bit) | Output |
|---------------|--------|
| 000           | 0      |
| 001           | 0      |
| 010           | 0      |
| 011           | 0      |
  
At this stage, we still cannot conclude whether the hidden function is constant or balanced. Therefore, one more function call is required to determine the answer with certainty. If the next input also returns $0$, then it is clearly a constant function; however, if it returns $1$, then we know for sure that it is a balanced function.

And similarly, **in the best case**, two queries to the oracle can determine if the hidden Boolean function, $f(x)$, is balanced: e.g. if we get both $f(0,0,0,...)\rightarrow 0$ and $f(1,0,0,...) \rightarrow 1$, then we know the function is balanced as we have obtained the two different outputs.  
</div>

### **Modeling the Problem with the Quantum Approach**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   

With the quantum approach, we only need to query the hidden function $f$ **once** to achieve our goal. Similar to the classical case, we also need to construct both a balanced and a constant version of the function within the quantum circuit, which we will be able to distinguish with just a single query!

The idea is straightforward: we use $n$ qubits as the input, along with one additional qubit representing the function’s output—often referred to as the output qubit. Therefore, our quantum circuit consists of a total of $n+1$ qubits.

<img src="img/djocircuit.png" alt="VS Code install prompt example" style="display: block; margin-left: auto; margin-right: auto; width: 45%; border: 1px solid #ccc; border-radius: 8px;">

**For the constant function**, we can design it as follows: for any input, either leave the output unchanged (meaning no operation is applied to the output qubit), or apply an $X$ (NOT) gate to the output qubit for every input. In both cases, the result is always the same (either always $0$ or always $1$) regardless of the input, which clearly captures the idea of a constant function.

**For the balanced function**, we must ensure its defining property: for all possible input cases, the outputs consist of exactly $50\%$ zeros and $50\%$ ones. To achieve this, we can simply apply CNOT gates, where the input qubits act as the control qubits and the output qubit is the target. Additionally, to increase the diversity in the ordering of the outputs, we may randomly apply $X$ gates on the input qubits and then cancel them after the balanced function is applied.

To make this more concrete, let us consider a simple example with $2$ input qubits and $1$ output qubit, as illustrated in the table below.

| Input (2 Qubits)| Output |
|-----------------|--------|
| 00              | 0      |
| 01              | 1      |
| 10              | 1      |
| 11              | 0      |

To increase the diversity in the ordering of the outputs, we apply **an $X$ gate to the first qubit**.

| Input (2 Qubits)| X-gate | Output |   
|-----------------|--------| ------ |   
| 00              | 10     | 1      |   
| 01              | 11     | 0      |   
| 10              | 00     | 0      |   
| 11              | 01     | 1      |   

</div>

## **Deliverable 1: Multiple oracle functions**

In [ ]:
# Number of input qubits
n_input = 5
# Declare quantum and classical registers for inputs, outputs and results
qreg_input = QuantumRegister(n_input, name="input")
qreg_output = QuantumRegister(1, name="output")
creg = ClassicalRegister(n_input, name="result")

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">   

**Task:** Implement at least one **constant** and one **balanced** oracle for n-bit input. Constant: output is always 0 or always 1. Balanced: exactly half of the $2^n$ inputs give 0 and half give 1.

Your task is to design a quantum circuit representing a Balanced Function, which ensures that the outputs always contain an equal number of 0s and 1s. At the same time, maintain diversity in the ordering of the output. Follow these steps:

1. For a randomly generated qubit string, apply X gates to the qubits marked as 1 in the string.
2. Apply CNOT gates using the input qubits as control qubits and the output qubit as the target.
3. Undo the X gates added in step 1 to preserve the correctness of the function.

</div>

### Constant oracle

In [ ]:
# Randomization for more diverse output behavior
output = rd.choice([0, 1])

constant_oracle = QuantumCircuit(qreg_input, qreg_output, creg)

### WRITE YOUR CODE BELOW THIS CELL ###



### YOUR CODE FINISHES HERE ###

### Balanced oracle (sample)


In [ ]:
# Generate a random qubit string; qubits with value 1 will be assigned an X gate
# The order from left to right corresponds to the output, input n−1, input n−2, input n−3, and so on.
qstr = format(rd.randint(1, 2**n_input - 1) , '0' + str(n_input) + 'b') 
qstr[::-1]  # Reverse the string to match qubit order

In [ ]:
balanced_oracle = QuantumCircuit(qreg_input, qreg_output, creg)

balanced_oracle.barrier()

### WRITE YOUR CODE BELOW THIS CELL ###



### YOUR CODE FINISHES HERE ###
balanced_oracle.barrier()

balanced_oracle.draw('mpl')

## **Deliverable 2: Working n-qubit Deutsch–Jozsa algorithm**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   
 
We assume that the function fed into the quantum circuit of the algorithm is **unknown**; it could be either a balanced function or a constant function, and the choice of which function to use is made **randomly**.
</div>

In [ ]:
# Assume that the function selection is performed randomly
oracle = rd.choice(["balanced", "constant"])

In [ ]:
oracle

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">   

**Task:** Build the full DJ circuit for **n** input qubits: initialise output to $|{-}\rangle$, apply H to all inputs, apply the oracle, apply H to all inputs, then measure the input qubits. Result: all zeros ⇒ constant; otherwise ⇒ balanced.

Your task is to **design an algorithm** to **determine** whether the function fed into the quantum circuit via the random function above is constant or balanced *(keep everything secret!)*. Follow these steps to construct the algorithm:

1. Create a quantum circuit using the existing registers.
2. Prepare the $|{-}\rangle$ state by applying an $X$ gate then $H$ to the output qubit.
3. Put all input qubits into a quantum superposition (apply $H$ to each).
4. Apply the oracle function (the hidden function) to the circuit.
5. Apply Hadamard gates to all input qubits after the oracle has been applied.
6. Apply measurement gates to all input qubits.

</div>

**Sample solution:** Below.

In [ ]:
dj_circuit = QuantumCircuit(qreg_input, qreg_output, creg)

### WRITE YOUR CODE BELOW THIS CELL ###




    
### YOUR CODE FINISHES HERE ###

dj_circuit.draw('mpl')

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">   
 
Great! After completing the steps above, we can see our quantum circuit fully constructed. By visually inspecting the simulated circuit, you might already get a sense of what kind of function it represents!
  
However, suppose the function is still hidden inside a **“black box.”** To determine its nature, we measure the classical outputs of the input qubits. As expected:
- If the function is constant, all measurements return $0$.
- If the function is balanced, all measurements return $1$.
</div>

## **Deliverable 6 (part 1): Ideal simulator**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">   

**Task:** Run the circuit on an ideal simulator and verify whether the function is constant or balanced.

Your task is to run the quantum algorithm on a simulator and read the results. Based on the measurements, determine whether the function is balanced or constant.

</div>

**Sample solution:** Below.

In [ ]:
### WRITE YOUR CODE BELOW THIS CELL ###


### YOUR CODE FINISHES HERE ###

plot_histogram(counts)

In [ ]:
# Check which oracle was used
if counts.get('0'*n_input, 0) == 1024:
    print("The oracle used was: constant")
else:
    print("The oracle used was: balanced")

## **Deliverable 3: Why the algorithm works — phase kickback**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">   

**Task:** Explain why the Deutsch–Jozsa algorithm correctly distinguishes constant from balanced oracles using the **phase kickback** mechanism.

</div>

**Your solution:**



## **Deliverable 4: Circuit depth and gate complexity as n scales**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">   

**Task:** For the n-qubit Deutsch–Jozsa circuit, analyse how **circuit depth** and **gate count** scale with **n**.

</div>

**Your solution:** 

---


In [ ]:
### WRITE YOUR CODE BELOW THIS CELL ###



### YOUR CODE FINISHES HERE ###

## **Deliverable 5: Quantum vs classical query complexity**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">   

**Task:** Compare the number of oracle queries needed in the classical and quantum settings.

</div>

**Your solution:**



## **Deliverable 6 (part 2): Noisy simulator**

<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify; color: #111; background-color: #fff8dc; padding: 15px; border-radius: 8px;">   

**Task:** Run the same Deutsch–Jozsa circuit on a **noisy** simulator and compare outcomes (e.g. success rate or measurement counts) with the ideal case.

</div>

**Your solution:** 

In [ ]:
### WRITE YOUR CODE BELOW THIS CELL ###



### YOUR CODE FINISHES HERE ###

## **Conclusion**

You have completed all six deliverables for the **Project: Deutsch–Jozsa Algorithm**:

1. **Working n-qubit implementation** of the algorithm  
2. **Multiple oracles** (constant and balanced)  
3. **Phase kickback** analysis explaining why the algorithm works  
4. **Circuit depth and gate complexity** as n scales  
5. **Quantum vs classical** query complexity comparison  
6. **Testing** on ideal and noisy simulators  

The Deutsch–Jozsa algorithm demonstrates **quantum advantage** with a single query versus up to \(2^{n-1}+1\) classical queries. In the next lab you can explore **Grover's algorithm** for searching unstructured data.

## **Conclusion**

You have completed all six deliverables for the **Deutsch–Jozsa algorithm** project: (1) n-qubit implementation, (2) constant and balanced oracles, (3) phase kickback analysis, (4) circuit depth and gate complexity vs n, (5) quantum vs classical query complexity, and (6) testing on ideal and noisy simulators. This algorithm is a foundational example of **quantum advantage**. Next you can explore **Grover's algorithm** for unstructured search.